# Carbon Accounting with libCBM Integration

This notebook demonstrates how to integrate `ws3` with `libcbm` for carbon accounting.

> **Prerequisites**: Completion of `070_ws3_quickstart_complete_workflow.ipynb`

## What You'll Learn

- How to set up a `ForestModel` from Woodstock data
- How to compile data for libCBM
- How to run libCBM simulations
- How to analyze carbon stocks and fluxes
- How to visualize carbon results

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import ws3.forest
from util import schedule_harvest_areacontrol, compile_scenario, plot_scenario

In [ ]:
# Model parameters
base_year = 2020
horizon = 10
period_length = 10
max_age = 1000
tvy_name = "totvol"

print(f"Model Parameters:")
print(f"  Base Year: {base_year}")
print(f"  Horizon: {horizon} periods")
print(f"  Period Length: {period_length} years")
print(f"  Max Age: {max_age} years")
print(f"  TVY Measure: {tvy_name}")

In [ ]:
fm = ws3.forest.ForestModel(model_name="tsa24_clipped",
                            model_path="data/woodstock_model_files_tsa24_clipped",
                            base_year=base_year,
                            horizon=horizon,
                            period_length=period_length,
                            max_age=max_age)
fm.import_landscape_section()
fm.import_areas_section(convert_periods_to_years=period_length)
fm.import_yields_section(convert_periods_to_years=period_length)
fm.import_actions_section(convert_periods_to_years=period_length)
fm.import_transitions_section(convert_periods_to_years=period_length)
fm.initialize_areas()
fm.add_null_action()
fm.reset_actions()

print(f"ForestModel loaded: {fm}")

In [ ]:
# Schedule some harvesting to create a realistic harvest schedule
sch = schedule_harvest_areacontrol(fm)

# Compile and display the harvest schedule
df = compile_scenario(fm)
fig, ax = plot_scenario(df)
plt.show()

In [ ]:
# Try to import libcbm
try:
    import libcbm
    print(f"libcbm version: {libcbm.__version__}")
except ImportError:
    print("libcbm not installed. Install with: pip install libcbm")
    print("This notebook requires libcbm for carbon accounting.")

In [ ]:
# Define species classifier
species_classifier_colname = "species"
leading_species_classifier_colname = "leading_species"

# Load CANFI species data
canfi_species = pd.read_csv("data/canfi_species.csv")
canfi_species.set_index("canfi_species", inplace=True)

print(f"Loaded {len(canfi_species)} CANFI species")

In [ ]:
# Compile yield data for libCBM
nv = 100  # number of age classes

data = {"theme0":[], "theme1":[], "theme2":[], "theme3":[], "theme4":[],
        species_classifier_colname:[], leading_species_classifier_colname:[], 
        **{"v%i" % i:[] for i in range(nv + 1)}}

leading_species_from_dtype_key = {}

for dtype_key, ytype, curves in fm.yields[:-1]:
    if ytype != "a": continue
    for yname, curve in curves:
        for i in range(5): data["theme%i" % i].append(dtype_key[i])
        species = "softwood" if int(yname[-4:]) < 1200 else "hardwood"
        leading_species_from_dtype_key[dtype_key] = species
        data[species_classifier_colname].append(species)
        data[leading_species_classifier_colname].append(species)
        for i in range(nv + 1): data["v%i" % i].append(curve[i * fm.period_length])

sit_yield = pd.DataFrame(data)
print(f"Compiled {len(sit_yield)} yield records")

In [ ]:
# Compile inventory data
names = ["_", "theme0", "theme1", "theme2", "theme3", "theme4", "age", "area"]
sit_inventory = pd.read_csv("data/woodstock_model_files_tsa24_clipped/tsa24_clipped.are",
                            delimiter=" ", header=None, names=names)
sit_inventory.drop('_', axis=1, inplace=True)

def _leading_species(dtype_key):
    for mask, leading_species in leading_species_from_dtype_key.items():
        if fm.match_mask(mask, dtype_key):
            return leading_species
        
def __leading_species(r):
    dtype_key = tuple(str(r["theme%i" % i]) for i in range(5))
    return _leading_species(dtype_key)

sit_inventory[species_classifier_colname] = sit_inventory.apply(__leading_species, axis=1)
sit_inventory["using_age_class"] = "FALSE"
sit_inventory["delay"] = 0
sit_inventory["landclass"] = 0
sit_inventory["historic_disturbance"] = "fire"
sit_inventory["last_pass_disturbance"] = sit_inventory.apply(
    lambda r: "fire" if r["theme2"] == r["theme4"] else "harvest", axis=1
)

print(f"Compiled {len(sit_inventory)} inventory records")

In [ ]:
# Compile classifiers
data = {"classifier_id":[], "name":[], "description":[]}
for i in range(5):
    data["classifier_id"].append(i+1)
    data["name"].append("_CLASSIFIER")
    data["description"].append("theme%i" % i)
    for v in fm.theme_basecodes(i):
        data["classifier_id"].append(i+1)
        data["name"].append(v)
        data["description"].append(v)

data["classifier_id"].append(6)
data["name"].append("_CLASSIFIER")
data["description"].append(species_classifier_colname)
data["classifier_id"].append(6)
data["name"].append("softwood")
data["description"].append("softwood")
data["classifier_id"].append(6)
data["name"].append("hardwood")
data["description"].append("hardwood")

sit_classifiers = pd.DataFrame(data)
print(f"Compiled {len(sit_classifiers)} classifiers")

In [ ]:
# Compile disturbance types
data = {"id":["harvest", "fire"],
        "name":["harvest", "fire"]}
sit_disturbance_types = pd.DataFrame(data)
print(f"Compiled disturbance types: {list(sit_disturbance_types['id'])}")

In [ ]:
# Compile age classes
data = {"name":["age_0"],
        "class_size":[0],
        "start_year":[0],
        "end_year":[0]}
for i, ac in enumerate(range(period_length, max_age+period_length, period_length)):
    data["name"].append("age_%i" % (i+1))
    data["class_size"].append(period_length)
    data["start_year"].append(ac - period_length + 1)
    data["end_year"].append(ac)
sit_age_classes = pd.DataFrame(data)
print(f"Compiled {len(sit_age_classes)} age classes")

In [ ]:
# Compile events from harvest schedule
columns = [
    "theme0", "theme1", "theme2", "theme3", "theme4",
    species_classifier_colname, "using_age_class",
    "min_softwood_age", "max_softwood_age",
    "min_hardwood_age", "max_hardwood_age",
    "MinYearsSinceDist", "MaxYearsSinceDist",
    "LastDistTypeID",
    "MinTotBiomassC", "MaxTotBiomassC",
    "MinSWMerchBiomassC", "MaxSWMerchBiomassC",
    "MinHWMerchBiomassC", "MaxHWMerchBiomassC",
    "MinTotalStemSnagC", "MaxTotalStemSnagC",
    "MinSWStemSnagC", "MaxSWStemSnagC",
    "MinHWStemSnagC", "MaxHWStemSnagC",
    "MinTotalStemSnagMerchC", "MaxTotalStemSnagMerchC",
    "MinSWMerchStemSnagC", "MaxSWMerchStemSnagC",
    "MinHWMerchStemSnagC", "MaxHWMerchStemSnagC",
    "efficiency", "sort_type", "target_type", "target",
    "disturbance_type", "disturbance_year"
]

data = {c:[] for c in columns}

for dtype_key, age, area, acode, period, _ in sch:
    for i in range(5): data["theme%i" % i].append(dtype_key[i])
    data[species_classifier_colname].append(_leading_species(dtype_key))
    data["using_age_class"].append("FALSE")
    for c in columns[7:29]: data[c].append(-1)
    data["efficiency"].append(1)
    data["sort_type"].append(3)  # oldest first
    data["target_type"].append("A")  # area target
    data["target"].append(area)
    data["disturbance_type"].append(acode)
    data["disturbance_year"].append(period * fm.period_length)

sit_events = pd.DataFrame(data)
print(f"Compiled {len(sit_events)} disturbance events")

In [ ]:
# Compile transitions from harvest action
au_table = pd.read_csv("data/au_table.csv")
au_table1 = au_table.set_index("au_id")
au_table2 = au_table.set_index("managed_curve_id")

columns = ["theme0", "theme1", "theme2", "theme3", "theme4",
           species_classifier_colname, "using_age_class",
           "min_softwood_age", "max_softwood_age",
           "min_hardwood_age", "max_hardwood_age",
           "disturbance_type",
           "to_theme0", "to_theme1", "to_theme2", "to_theme3", "to_theme4",
           "to_%s" % species_classifier_colname,
           "regen_delay", "reset_age", "percent"]

data = {c:[] for c in columns}

for acode in fm.transitions:
    if acode != "harvest": continue
    for smask in fm.transitions[acode]:
        tmask, tprop, _, _, _, _, _ = fm.transitions[acode][smask][""][0]
        for i in range(5): data["theme%i" % i].append(smask[i])
        data[species_classifier_colname].append(
            "softwood" if au_table1.loc[int(smask[2])].canfi_species < 1200 else "hardwood"
        )
        data["using_age_class"].append("FALSE")
        for c in columns[7:11]: data[c].append(-1)
        data["disturbance_type"].append("harvest")
        for i in range(5): data["to_theme%i" % i].append(tmask[i])
        data["to_%s" % species_classifier_colname].append(
            "softwood" if au_table2.loc[int(tmask[4])].canfi_species < 1200 else "hardwood"
        )
        data["regen_delay"].append(0)
        data["reset_age"].append(0)
        data["percent"].append(100)

sit_transitions = pd.DataFrame(data)
print(f"Compiled {len(sit_transitions)} transitions")

In [ ]:
# Import libCBM modules
from libcbm.input.sit import sit_reader
from libcbm.input.sit import sit_cbm_factory

In [ ]:
# Compile SIT data
sit_data = sit_reader.parse(
    sit_classifiers=sit_classifiers,
    sit_disturbance_types=sit_disturbance_types,
    sit_age_classes=sit_age_classes,
    sit_inventory=sit_inventory,
    sit_yield=sit_yield,
    sit_events=sit_events,
    sit_transitions=sit_transitions,
    sit_eligibilities=None
)
print("SIT data compiled successfully")

In [ ]:
# Create SIT config
sit_config = {
    "mapping_config": {
        "nonforest": None,
        "species": {
            "species_classifier": species_classifier_colname,
            "species_mapping": [
                {"user_species": "softwood", "default_species": "Softwood forest type"},
                {"user_species": "hardwood", "default_species": "Hardwood forest type"}
            ]
        },
        "spatial_units": {
            "mapping_mode": "SingleDefaultSpatialUnit",
            "admin_boundary": "British Columbia",
            "eco_boundary": "Montane Cordillera"},
        "disturbance_types": {
            "disturbance_type_mapping": [
                {"user_dist_type": "harvest", "default_dist_type": "Clearcut harvesting without salvage"},
                {"user_dist_type": "fire", "default_dist_type": "Wildfire"}
            ]
        }
    }
}
print("SIT config created")

In [ ]:
# Initialize SIT and CBM
sit = sit_cbm_factory.initialize_sit(sit_data=sit_data, config=sit_config)
classifiers, inventory = sit_cbm_factory.initialize_inventory(sit)

print(f"Classifiers: {len(classifiers)}")
print(f"Inventory records: {len(inventory)}")

In [ ]:
# Create CBM output collector
from libcbm.model.cbm.cbm_output import CBMOutput

cbm_output = CBMOutput(
    classifier_map=sit.classifier_value_names,
    disturbance_type_map=sit.disturbance_name_map
)
print("CBM output collector created")

In [ ]:
# Import CBM simulator
from libcbm.storage.backends import BackendType
from libcbm.model.cbm import cbm_simulator

In [ ]:
# Run CBM simulation
with sit_cbm_factory.initialize_cbm(sit) as cbm:
    rule_based_processor = sit_cbm_factory.create_sit_rule_based_processor(sit, cbm)
    
    cbm_simulator.simulate(
        cbm,
        n_steps=200,
        classifiers=classifiers,
        inventory=inventory,
        pre_dynamics_func=rule_based_processor.pre_dynamics_func,
        reporting_func=cbm_output.append_simulation_result,
        backend_type=BackendType.numpy
    )
    
    print("CBM simulation completed")

In [ ]:
# Analyze carbon stocks over time
pi = cbm_output.classifiers.to_pandas().merge(
    cbm_output.pools.to_pandas(), 
    left_on=["identifier", "timestep"], 
    right_on=["identifier", "timestep"]
)

# Define carbon pools
biomass_pools = [
    "SoftwoodMerch", "SoftwoodFoliage", "SoftwoodOther", 
    "SoftwoodCoarseRoots", "SoftwoodFineRoots",
    "HardwoodMerch", "HardwoodFoliage", "HardwoodOther", 
    "HardwoodCoarseRoots", "HardwoodFineRoots"
]

dom_pools = [
    "AboveGroundVeryFastSoil", "BelowGroundVeryFastSoil",
    "AboveGroundFastSoil", "BelowGroundFastSoil",
    "MediumSoil", "AboveGroundSlowSoil", "BelowGroundSlowSoil",
    "SoftwoodStemSnag", "SoftwoodBranchSnag",
    "HardwoodStemSnag", "HardwoodBranchSnag"
]

# Calculate annual carbon stocks
annual_carbon_stocks = pd.DataFrame({
    "Year": pi["timestep"],
    "Biomass": pi[biomass_pools].sum(axis=1),
    "DOM": pi[dom_pools].sum(axis=1),
    "Total Ecosystem": pi[biomass_pools+dom_pools].sum(axis=1)
})

# Plot carbon stocks over time
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Biomass carbon
axes[0].plot(annual_carbon_stocks["Year"], annual_carbon_stocks["Biomass"], 'b-', linewidth=2)
axes[0].set_title('Biomass Carbon Stock', fontsize=14)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Carbon (t C/ha)')
axes[0].grid(True, alpha=0.3)

# Total ecosystem carbon
axes[1].plot(annual_carbon_stocks["Year"], annual_carbon_stocks["Total Ecosystem"], 'g-', linewidth=2)
axes[1].set_title('Total Ecosystem Carbon Stock', fontsize=14)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Carbon (t C/ha)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze carbon fluxes
fi = cbm_output.flux.to_pandas()

# Define key flux indicators
annual_process_fluxes = [
    "DecayDOMCO2Emission",
    "DeltaBiomass_AG", "DeltaBiomass_BG",
    "TurnoverMerchLitterInput", "TurnoverFolLitterInput",
    "TurnoverOthLitterInput", "TurnoverCoarseLitterInput",
    "TurnoverFineLitterInput",
    "DecayVFastAGToAir", "DecayVFastBGToAir",
    "DecayFastAGToAir", "DecayFastBGToAir",
    "DecayMediumToAir", "DecaySlowAGToAir", "DecaySlowBGToAir",
    "DecaySWStemSnagToAir", "DecaySWBranchSnagToAir",
    "DecayHWStemSnagToAir", "DecayHWBranchSnagToAir"
]

# Plot net ecosystem exchange
net_flux = fi[["timestep"] + annual_process_fluxes].groupby("timestep").sum()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(net_flux.index, net_flux["DeltaBiomass_AG"] + net_flux["DeltaBiomass_BG"], 
       'b-', linewidth=2, label='Biomass Change')
ax.plot(net_flux.index, -net_flux["DecayDOMCO2Emission"], 
       'r-', linewidth=2, label='Decomposition CO2')
ax.set_title('Net Ecosystem Exchange', fontsize=14)
ax.set_xlabel('Year')
ax.set_ylabel('Carbon Flux (t C/ha/year)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate carbon budget
initial_carbon = annual_carbon_stocks["Total Ecosystem"].iloc[0]
final_carbon = annual_carbon_stocks["Total Ecosystem"].iloc[-1]
net_carbon_change = final_carbon - initial_carbon

print(f"Initial Carbon Stock: {initial_carbon:.2f} t C/ha")
print(f"Final Carbon Stock: {final_carbon:.2f} t C/ha")
print(f"Net Carbon Change: {net_carbon_change:.2f} t C/ha")
print(f"Average Annual Sequestration: {net_carbon_change/200:.4f} t C/ha/year")

# Export results to CSV
annual_carbon_stocks.to_csv("carbon_stocks_results.csv", index=False)
print("\nResults exported to carbon_stocks_results.csv")

In [ ]:
# Summary of carbon accounting results
print("=" * 60)
print("CARBON ACCOUNTING SUMMARY")
print("=" * 60)
print(f"Simulation Period: 200 years")
print(f"Initial Biomass Carbon: {annual_carbon_stocks['Biomass'].iloc[0]:.2f} t C/ha")
print(f"Final Biomass Carbon: {annual_carbon_stocks['Biomass'].iloc[-1]:.2f} t C/ha")
print(f"Initial DOM Carbon: {annual_carbon_stocks['DOM'].iloc[0]:.2f} t C/ha")
print(f"Final DOM Carbon: {annual_carbon_stocks['DOM'].iloc[-1]:.2f} t C/ha")
print(f"Net Ecosystem Carbon Change: {net_carbon_change:.2f} t C/ha")
print("=" * 60)

In [ ]:
# This notebook demonstrated carbon accounting with libCBM integration
print("Carbon accounting notebook complete!")
print("Next: Try 073_ws3_spatial_constraints.ipynb for spatial analysis")